In [1]:

%pip install pandas numpy spacy matplotlib plotly scikit-learn beautifulsoup4 nltk fastopic optuna optuna-dashboard

Note: you may need to restart the kernel to use updated packages.


In [2]:

import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from fastopic import FASTopic
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from wordcloud import WordCloud

## FASTopic

In [3]:
# Carga de datos
test = pd.read_csv("data/research-articles/test.csv")
train = pd.read_csv("data/research-articles/train.csv")

display(test.head())
display(train.head())

,ID,TITLE,ABSTRACT
0,20973,Closed-form Marginal Likelihood in Gamma-Poiss...,We present novel understandings of the Gamma...
1,20974,Laboratory mid-IR spectra of equilibrated and ...,Meteorites contain minerals from Solar Syste...
2,20975,Case For Static AMSDU Aggregation in WLANs,Frame aggregation is a mechanism by which mu...
3,20976,The $Gaia$-ESO Survey: the inner disk intermed...,Milky Way open clusters are very diverse in ...
4,20977,Witness-Functions versus Interpretation-Functi...,Proving that a cryptographic protocol is cor...


,ID,TITLE,ABSTRACT,Computer Science,Physics,Mathematics,Statistics,Quantitative Biology,Quantitative Finance
0,1,Reconstructing Subject-Specific Effect Maps,Predictive models allow subject-specific inf...,1,0,0,0,0,0
1,2,Rotation Invariance Neural Network,Rotation invariance and translation invarian...,1,0,0,0,0,0
2,3,Spherical polyharmonics and Poisson kernels fo...,We introduce and develop the notion of spher...,0,0,1,0,0,0
3,4,A finite element approximation for the stochas...,The stochastic Landau--Lifshitz--Gilbert (LL...,0,0,1,0,0,0
4,5,Comparative study of Discrete Wavelet Transfor...,Fourier-transform infra-red (FTIR) spectra o...,1,0,0,1,0,0


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20972 entries, 0 to 20971
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ID                    20972 non-null  int64 
 1   TITLE                 20972 non-null  object
 2   ABSTRACT              20972 non-null  object
 3   Computer Science      20972 non-null  int64 
 4   Physics               20972 non-null  int64 
 5   Mathematics           20972 non-null  int64 
 6   Statistics            20972 non-null  int64 
 7   Quantitative Biology  20972 non-null  int64 
 8   Quantitative Finance  20972 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 1.4+ MB


In [5]:
train.dtypes

ID                       int64
TITLE                   object
ABSTRACT                object
Computer Science         int64
Physics                  int64
Mathematics              int64
Statistics               int64
Quantitative Biology     int64
Quantitative Finance     int64
dtype: object

In [24]:
train_small = train.sample(n=10, random_state=42)
documents = list(train_small.ABSTRACT.values)  # utilizaremos solo la columan Abstract como los documentos
print(documents[0])  # como ejemplo imprimimos el primer documento

  Layer normalization is a recently introduced technique for normalizing the
activities of neurons in deep neural networks to improve the training speed and
stability. In this paper, we introduce a new layer normalization technique
called Dynamic Layer Normalization (DLN) for adaptive neural acoustic modeling
in speech recognition. By dynamically generating the scaling and shifting
parameters in layer normalization, DLN adapts neural acoustic models to the
acoustic variability arising from various factors such as speakers, channel
noises, and environments. Unlike other adaptive acoustic models, our proposed
approach does not require additional adaptation data or speaker information
such as i-vectors. Moreover, the model size is fixed as it dynamically
generates adaptation parameters. We apply our proposed DLN to deep
bidirectional LSTM acoustic models and evaluate them on two benchmark datasets
for large vocabulary ASR experiments: WSJ and TED-LIUM release 2. The
experimental results s

In [37]:
clean_text = documents
tokenized_texts = [doc.split() for doc in clean_text]
print(tokenized_texts[0])
dictionary = Dictionary(tokenized_texts)
dictionary.filter_extremes(no_below=5, no_above=0.8)

['Layer', 'normalization', 'is', 'a', 'recently', 'introduced', 'technique', 'for', 'normalizing', 'the', 'activities', 'of', 'neurons', 'in', 'deep', 'neural', 'networks', 'to', 'improve', 'the', 'training', 'speed', 'and', 'stability.', 'In', 'this', 'paper,', 'we', 'introduce', 'a', 'new', 'layer', 'normalization', 'technique', 'called', 'Dynamic', 'Layer', 'Normalization', '(DLN)', 'for', 'adaptive', 'neural', 'acoustic', 'modeling', 'in', 'speech', 'recognition.', 'By', 'dynamically', 'generating', 'the', 'scaling', 'and', 'shifting', 'parameters', 'in', 'layer', 'normalization,', 'DLN', 'adapts', 'neural', 'acoustic', 'models', 'to', 'the', 'acoustic', 'variability', 'arising', 'from', 'various', 'factors', 'such', 'as', 'speakers,', 'channel', 'noises,', 'and', 'environments.', 'Unlike', 'other', 'adaptive', 'acoustic', 'models,', 'our', 'proposed', 'approach', 'does', 'not', 'require', 'additional', 'adaptation', 'data', 'or', 'speaker', 'information', 'such', 'as', 'i-vectors.

In [27]:
def coherence_from_topics(topic_words, tokenized_texts, dictionary):
    if not topic_words:
        return -1e6
    try:
        cm = CoherenceModel(topics=topic_words, texts=tokenized_texts, dictionary=dictionary, coherence='c_v')
        return float(cm.get_coherence())
    except Exception as e:
        print("Error coherence:", e)
        return -1e6

In [53]:
def get_top_words(top_words_out, n_words=10):
    normalized_topics = []
    for topic in top_words_out:
        words = []
        for entry in topic:
            # 1) si es tupla (word,score)
            if isinstance(entry, tuple) and len(entry) >= 1:
                candidate = entry[0]
            else:
                candidate = entry

            # 2) si candidate es lista/tupla de caracteres -> unir sin separador
            if isinstance(candidate, (list, tuple)) and all(isinstance(ch, str) and len(ch) == 1 for ch in candidate):
                s = ''.join(candidate)
            # 3) si es string -> usar tal cual
            elif isinstance(candidate, str):
                s = candidate
            else:
                # fallback
                s = str(candidate)

            # 4) limpia y separa en palabras (por espacios)
            #    Puede ser una frase completa, por eso split()
            parts = s.strip().split()
            for p in parts:
                if p:
                    words.append(p)
                if len(words) >= n_words:
                    break
            if len(words) >= n_words:
                break

        # si no se extrajo nada, insertar token vacío para no romper gensim
        if not words:
            words = [""]

        normalized_topics.append(words[:n_words])

    return normalized_topics

In [54]:
def objective(trial):
    """
    Construye un FASTopic con los hiperparámetros sugeridos, entrena (fit_transform)
    y devuelve coherencia c_v (a maximizar).
    """
    # Hiperparámetros a buscar
    n_topics = trial.suggest_int("num_topics", 10, 80)  # rango típico
    DT_alpha = trial.suggest_categorical("DT_alpha", [1.0, 5.0, 10.0, 15.0])
    normalize_embeddings = trial.suggest_categorical("normalize_embeddings", [True, False])
    low_memory = trial.suggest_categorical("low_memory", [False, True])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-3, 1e-1)
    epochs = trial.suggest_int("epochs", 20, 200)

    # build model (constructor flexible según versión)
    try:
        # FASTopic puede aceptar: FASTopic(num_topics, preprocess=..., doc_embed_model=..., normalize_embeddings=...)
        # Aquí usamos constructor con num_topics y algunos flags; si tu versión tiene firma distinta, se adapta por try/except
        model = FASTopic(num_topics=n_topics,
                         normalize_embeddings=normalize_embeddings,
                         DT_alpha=DT_alpha,
                         low_memory=low_memory,
                         verbose=False)
    except TypeError:
        # fallback: construir sin algunos argumentos
        model = FASTopic(num_topics=n_topics, verbose=False)

    # Entrenar (fit_transform devuelve top_words y doc_topic_dist según README)

    t0 = time.time()
    top_words_out, doc_topic_dist = model.fit_transform(clean_text, learning_rate=learning_rate, epochs=epochs)
    elapsed = time.time() - t0

    # signature diferente: intentar sin lr/epochs

    t0 = time.time()
    top_words_out, doc_topic_dist = model.fit_transform(clean_text)
    elapsed = time.time() - t0

    if len(top_words_out) == 0:
        trial.set_user_attr("n_topics_found", 0)
        return -1e6

    topics_for_gensim = get_top_words(top_words_out)
    print("Top words found:", topics_for_gensim)

    # calcular coherencia c_v
    coh = coherence_from_topics(topics_for_gensim, tokenized_texts, dictionary)

    # guardar attrs útiles
    trial.set_user_attr("elapsed_s", elapsed)
    trial.set_user_attr("n_topics_found", len(top_words))

    return float(coh)


In [55]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
def run_optuna_fastopic(n_trials=20):
    sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    print("Iniciando Optuna para FASTopic. n_trials =", n_trials)
    study.optimize(objective, n_trials=n_trials)
    print("Mejor coherencia:", study.best_value)
    print("Mejores hiperparámetros:", study.best_params)
    return study

In [56]:
study = run_optuna_fastopic(n_trials=20)

[I 2025-12-03 11:53:38,355] A new study created in memory with name: no-name-6cd21de4-b31a-41b5-9c5b-e7149b081481
C:\Users\zeldan\AppData\Local\Temp\ipykernel_15740\3220774820.py:11: FutureWarning:

suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.



Iniciando Optuna para FASTopic. n_trials = 20


parsing texts: 100%|██████████| 10/10 [00:00<00:00, 9998.34it/s]
C:\Users\zeldan\anaconda3\envs\tfg-topics2\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'

Training FASTopic: 100%|██████████| 200/200 [00:08<00:00, 24.81it/s]
[I 2025-12-03 11:53:50,925] Trial 0 finished with value: -1000000.0 and parameters: {'num_topics': 36, 'DT_alpha': 1.0, 'normalize_embeddings': True, 'low_memory': False, 'learning_rate': 0.02607024758370768, 'epochs': 23}. Best is trial 0 with value: -1000000.0.


Top words found: ['p r o b l e m s   g r a p h   n o d e   m o d e l   l o w e r   a r b i t r a r y   s t e p   g e n e r a t i n g   n e w   r e q u i r e   b a s e d   l i m i t   r o u t i n g   c o n s i d e r   g e n e r a t e d', 'p r o p o s e d   a p p r o a c h   a d a p t i v e   f i x e d   n e w   i m p r o v e   g e n e r a t i n g   b a s e d   d a t a   l i m i t   m e t h o d   p a p e r   m o d e l   r e q u i r e   t r a i n i n g', 'a l g o r i t h m i c   m o d u l a r i t y   m a x i m i z a t i o n   p o p u l a r   i n s i g h t   p r o v i d e s   g r e e d y   a l g o r i t h m s   s t a t e s   a c h i e v e d   i s s u e   p e r s i s t e n t l y   c o m m u n i t y   u p d a t e   i m p l e m e n t a t i o n', 'a l g o r i t h m i c   m o d u l a r i t y   p o p u l a r   m a x i m i z a t i o n   i n s i g h t   g r e e d y   a l g o r i t h m s   c o m m u n i t y   a c h i e v e d   s t a t e s   p e r s i s t e n t l y   a p a r t   b e t t e r   i m p 

Training FASTopic: 100%|██████████| 200/200 [00:12<00:00, 16.15it/s]
[I 2025-12-03 11:54:09,400] Trial 1 finished with value: -1000000.0 and parameters: {'num_topics': 78, 'DT_alpha': 1.0, 'normalize_embeddings': False, 'low_memory': False, 'learning_rate': 0.01673808578875214, 'epochs': 45}. Best is trial 0 with value: -1000000.0.
[W 2025-12-03 11:54:09,402] Trial 2 failed with parameters: {'num_topics': 30, 'DT_alpha': 10.0, 'normalize_embeddings': False, 'low_memory': True, 'learning_rate': 0.002193048555664369, 'epochs': 31} because of the following error: AssertionError().
Traceback (most recent call last):
  File "C:\Users\zeldan\anaconda3\envs\tfg-topics2\Lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\zeldan\AppData\Local\Temp\ipykernel_15740\3220774820.py", line 30, in objective
    top_words_out, doc_topic_dist = model.fit_transform(clean_text, learning_rate=learning_rat

Top words found: ['c o l o r i n g   p e t t i e   r a n d o m i z e d   l c l   s q r t   c h u n g   r o u n d   l l l   l o g   b o u n d   i m p r o v e m e n t   i n c l u d e   c h a n g   r o u n d s   d i s t r i b u t e d', 'd a t a   p a p e r   c l a s s i f i c a t i o n   o u t l i e r s   c l u s t e r i n g   b a y e s i a n   m i x t u r e   m i s s i n g   v a r i a b l e s   l a t e n t   h a n d l e   i n f e r e n c e   p o s s i b i l i t y   i n t r o d u c t i o n   s u p e r v i s e d', 'a l g e b r a   k o s z u l   t h e o r y   b a c k g r o u n d   s u p e r g r a v i t y   p r e s e n t e d   s h o w n   d u a l   e x p l i c i t l y   p e r t u r b a t i o n   g r a v i t a t i o n a l   s u p e r s y m m e t r i c   d u a l i t y   o r d e r s   a u t h o r', 'c o d e   p u l s e d y n   n o n l i n e a r   m o r s e   j o n e s   s o l i t o n   b o d y   i n t e g r a b l e   t s i n g o u   l e n n a r d   f e r m i   p a s t a   d y n a m i c s   s p 

AssertionError: 

In [31]:

# ========== Reentrenar el mejor modelo con los mejores hiperparámetros ==========
best = study.best_params
print("Reentrenando FASTopic con:", best)

# Intentar crear modelo con esos parámetros

best_model = FASTopic(num_topics=best["num_topics"],
                      normalize_embeddings=best.get("normalize_embeddings", True),
                      DT_alpha=best.get("DT_alpha", 5.0),
                      low_memory=best.get("low_memory", False),
                      device=best.get("device", "cpu"),
                      verbose=True)


# Fit final (intentar con lr/epochs)

best_top_words, best_doc_topic_dist = best_model.fit_transform(clean_text, learning_rate=best.get("learning_rate", 0.01), epochs=best.get("epochs", 100))


[I 2025-12-03 11:31:41,432] A new study created in memory with name: no-name-e150bcbe-393f-4732-8b84-420130c8960f


Iniciando Optuna para FASTopic. n_trials = 20


C:\Users\zeldan\AppData\Local\Temp\ipykernel_15740\3436640344.py:11: FutureWarning:

suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.

parsing texts: 100%|██████████| 10/10 [00:00<00:00, 9991.20it/s]
C:\Users\zeldan\anaconda3\envs\tfg-topics2\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'

Training FASTopic: 100%|██████████| 200/200 [00:06<00:00, 31.55it/s]
[I 2025-12-03 11:31:52,088] Trial 0 finished with value: -1000000.0 and parameters: {'num_topics': 36, 'DT_alpha': 1.0, 'normalize_embeddings': True, 'low_memory': False, 'learning_rate': 0.02607024758370768, 'epochs': 23}. Best is trial 0 with value: -1000000.0.


Error coherence: unable to interpret topic as either a list of tokens or a list of ids


Training FASTopic: 100%|██████████| 200/200 [00:09<00:00, 20.81it/s]
[I 2025-12-03 11:32:06,860] Trial 1 finished with value: -1000000.0 and parameters: {'num_topics': 78, 'DT_alpha': 1.0, 'normalize_embeddings': False, 'low_memory': False, 'learning_rate': 0.01673808578875214, 'epochs': 45}. Best is trial 0 with value: -1000000.0.
[W 2025-12-03 11:32:06,862] Trial 2 failed with parameters: {'num_topics': 30, 'DT_alpha': 10.0, 'normalize_embeddings': False, 'low_memory': True, 'learning_rate': 0.002193048555664369, 'epochs': 31} because of the following error: AssertionError().
Traceback (most recent call last):
  File "C:\Users\zeldan\anaconda3\envs\tfg-topics2\Lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\zeldan\AppData\Local\Temp\ipykernel_15740\3436640344.py", line 30, in objective
    top_words_out, doc_topic_dist = model.fit_transform(clean_text, learning_rate=learning_rat

Error coherence: unable to interpret topic as either a list of tokens or a list of ids


AssertionError: 

In [42]:
best_model.get_top_words()

Topic 0: magnetic phase spin states range temperature orders coupling strong weak critical length transition mode nearest
Topic 1: support needed established protocol years path challenges internet issue plays multiplex lce secure exploits codeword
Topic 2: model models order distribution density parameters proposed process noise error compared standard mean probability rate
Topic 3: able frames flow games motion action potentially robot sensor shape environment robots super parallel environments
Topic 4: quantum frequency spectrum response experimentally measurement account experiment frequencies exciton freedom pulse damping appropriate phonon
Topic 5: study field systems large different dimensional properties energy small obtained interactions observed including type like
Topic 6: control range contrast stability evolutionary perspective principles pair drift takes vehicle kinetic theoretic intrinsic utility
Topic 7: approximation samples bayesian inference setting estimation kernel

['magnetic phase spin states range temperature orders coupling strong weak critical length transition mode nearest',
 'support needed established protocol years path challenges internet issue plays multiplex lce secure exploits codeword',
 'model models order distribution density parameters proposed process noise error compared standard mean probability rate',
 'able frames flow games motion action potentially robot sensor shape environment robots super parallel environments',
 'quantum frequency spectrum response experimentally measurement account experiment frequencies exciton freedom pulse damping appropriate phonon',
 'study field systems large different dimensional properties energy small obtained interactions observed including type like',
 'control range contrast stability evolutionary perspective principles pair drift takes vehicle kinetic theoretic intrinsic utility',
 'approximation samples bayesian inference setting estimation kernel convergence guarantees signals negative s

In [ ]:
fig = best_model.visualize_topic_hierarchy()
fig.show()

In [36]:
topic_ids, topic_words = extract_top_words_from_fastopic(best_model, top_n=10)
print(topic_words)
df_topics = pd.DataFrame({
    "topic_id": topic_ids,
    "top_words": ["".join(ws) for ws in topic_words]
})
display(df_topics.head(40))

Topic 0: magnetic phase spin states range temperature orders coupling strong weak critical length transition mode nearest
Topic 1: support needed established protocol years path challenges internet issue plays multiplex lce secure exploits codeword
Topic 2: model models order distribution density parameters proposed process noise error compared standard mean probability rate
Topic 3: able frames flow games motion action potentially robot sensor shape environment robots super parallel environments
Topic 4: quantum frequency spectrum response experimentally measurement account experiment frequencies exciton freedom pulse damping appropriate phonon
Topic 5: study field systems large different dimensional properties energy small obtained interactions observed including type like
Topic 6: control range contrast stability evolutionary perspective principles pair drift takes vehicle kinetic theoretic intrinsic utility
Topic 7: approximation samples bayesian inference setting estimation kernel

,topic_id,top_words
0,0,magnetic p
1,1,support ne
2,2,model mode
3,3,able frame
4,4,quantum fr
5,5,study fiel
6,6,control ra
7,7,approximat
8,8,simple for
9,9,informatio


In [25]:
best_model.visualize_topic(top_n=8)

In [39]:
# Fix 100% robusto para tus top words
def reconstruct_words_from_char_lists(topic_token_list):
    words = []
    for token_list in topic_token_list:
        # unir caracteres
        joined = "".join(token_list).strip()
        # dividir si hay espacios dentro
        for w in joined.split(" "):
            w = w.strip()
            if len(w) > 2:
                words.append(w)
    return words

# reconstruir todos los tópicos
fixed_topics = [reconstruct_words_from_char_lists(topic) for topic in topic_words]
print(fixed_topics)

# WordCloud
from wordcloud import WordCloud
import matplotlib.pyplot as plt

for tid, words in enumerate(fixed_topics):
    if not words:
        print("Topic vacío:", tid)
        continue

    text = " ".join(words)
    wc = WordCloud(width=1000, height=400, background_color="white", collocations=False).generate(text)

    plt.figure(figsize=(10,4))
    plt.imshow(wc, interpolation="bilinear")
    plt.title(f"Topic {tid}")
    plt.axis("off")
    plt.show()


[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
Topic vacío: 0
Topic vacío: 1
Topic vacío: 2
Topic vacío: 3
Topic vacío: 4
Topic vacío: 5
Topic vacío: 6
Topic vacío: 7
Topic vacío: 8
Topic vacío: 9
Topic vacío: 10
Topic vacío: 11
Topic vacío: 12
Topic vacío: 13
Topic vacío: 14
Topic vacío: 15
Topic vacío: 16
Topic vacío: 17
Topic vacío: 18
Topic vacío: 19
Topic vacío: 20
Topic vacío: 21
Topic vacío: 22
Topic vacío: 23
Topic vacío: 24
Topic vacío: 25
Topic vacío: 26
Topic vacío: 27
Topic vacío: 28
Topic vacío: 29
Topic vacío: 30
Topic vacío: 31
Topic vacío: 32
Topic vacío: 33
Topic vacío: 34
Topic vacío: 35
